# Models & Observations — Usage Example

This notebook shows a practical example of comparing observations and model predictions, e.g. deriving age and metallicity from grid models. We use random SDSS spectra from red galaxies that have both measured velocity dispersion and redshifts. Using this information we can de-redshift and broaden to a common resolution and measure the indices in the observed spectra. Also, because they have an associated error spectrum, we can estimate the errors in the index measurements by simulating different spectra.

In [ ]:
from saira import saira
import numpy as np
import matplotlib.pyplot as plt
import os

# Resolve paths to the bundled example data
import saira as saira_pkg
examples_dir = os.path.join(os.path.dirname(saira_pkg.__file__), 'examples')
suport_dir   = os.path.join(os.path.dirname(saira_pkg.__file__), 'suport_files')
print('Examples dir:', examples_dir)
print('Support dir:', suport_dir)

## 1. Measure indices in SDSS spectra

We measure the indices in the SDSS spectra, broadening them to σ = 300 km/s and using 100 MC iterations for error estimation.

In [ ]:
sdss_spectra = saira(
    filename=os.path.join(examples_dir, 'sdss_table_example.dat'),
    path_to_files=os.path.join(examples_dir, 'sdss_example'),
    IndexDefs=os.path.join(suport_dir, 'less_defs.ind'),
    sigma_fin=300,
    compute_idx=os.path.join(examples_dir, 'compute.txt'),
    print_log='sdss_log.txt',
    simulate=100
)
sdss_spectra.head()

In [ ]:
# Propagate MgFe' error
sdss_spectra['e_MgFe_prime'] = np.sqrt(
    (sdss_spectra['MgFe_prime']*sdss_spectra['e_Mgb']/sdss_spectra['Mgb'])**2 + 
    ((sdss_spectra['Mgb']/sdss_spectra['MgFe_prime'])**2)*((0.72*sdss_spectra['e_Fe5270'])**2+(0.28*sdss_spectra['e_Fe5335'])**2)
)/2.0

## 2. Measure the same indices in E-MILES models

Now we measure the same indices in the E-MILES models, broadening them from their native FWHM = 2.51 Å to σ = 300 km/s, matching the broadening established for the data.

In [ ]:
miles_models = saira(
    filename=os.path.join(examples_dir, 'miles_table.dat'),
    path_to_files=os.path.join(examples_dir, 'sdss_example', 'models'),
    IndexDefs=os.path.join(suport_dir, 'less_defs.ind'),
    FWHM_ini=2.51,
    sigma_fin=300,
    compute_idx=os.path.join(examples_dir, 'compute.txt'),
    print_log='models_log.txt'
)
miles_models.head()

## 3. Build the grid and derive parameters

The `pandas` functionality makes it easy to slice, build the grid, derive parameters, and plot the data.

In [ ]:
data_points = sdss_spectra[['MgFe_prime','Hbeta_o']]
e_data_points = sdss_spectra[['e_MgFe_prime','e_Hbeta_o']]

In [ ]:
from scipy.interpolate import LinearNDInterpolator

x = miles_models['Mgb']
y = miles_models['Hbeta_o']
age = miles_models['age']
met = miles_models['met']

age_interp = LinearNDInterpolator(list(zip(x, y)), age)
met_interp = LinearNDInterpolator(list(zip(x, y)), met)

sdss_spectra['age'] = age_interp(data_points)
sdss_spectra['met'] = met_interp(data_points)

print('Derived ages and metallicities:')
sdss_spectra[['age', 'met']]

## 4. Plot the diagnostic diagram

Now we plot the data on the model grid.

In [ ]:
grid = miles_models.pivot(columns='age', index='met', values=['Hbeta_o', 'MgFe_prime'])

age_to_plot = [0.5, 1.0, 3.0, 5.0, 14.0]
met_to_plot = [-1.49, -0.66, 0.06, 0.40]

In [ ]:
plt.plot(grid['MgFe_prime'][age_to_plot].loc[met_to_plot],
         grid['Hbeta_o'][age_to_plot].loc[met_to_plot], 'k-', lw=1)
plt.plot(grid['MgFe_prime'][age_to_plot].loc[met_to_plot].T,
         grid['Hbeta_o'][age_to_plot].loc[met_to_plot].T, 'k-', lw=1)
plt.errorbar(data_points['MgFe_prime'], data_points['Hbeta_o'],
              xerr=e_data_points['e_MgFe_prime'], yerr=e_data_points['e_Hbeta_o'], 
              fmt="ok", label="SDSS spectra")

plt.legend()
plt.xlabel(r"MgFe'")
plt.ylabel(r'H$\beta_o$')
plt.show()

Now comes the fun part! Try using your own data and matching with the models. Are the points affected by dust? Ionized gas? Do the models need extension?

Hope this tutorial was helpful — good luck interpreting your data.